# 03 — 2D slice model (Swin-T, ImageNet-pretrained) with 5-fold CV

1. **Scan-protocol confound check:** voxel spacing (z) per label and per site (AMC vs R01), to see whether acquisition protocol differs between classes.
2. **Final 2D run:** `swin_tiny_patch4_window7_224` (timm), last stages + head fine-tuned, 6 most informative axial slices per patient, patient-level probability = mean over slices, class-weighted loss + weighted sampler, 10 epochs, 5-fold CV. Two patients with corrupted volumes (R01-091, R01-133) are excluded → 170 patients.

Note: the best epoch of each fold is chosen on that fold's validation set (balanced accuracy + AUC), so the reported numbers are optimistic (see README, *Limitations*).

Paths are Windows paths from the original machine (`D:\\NSCLC_PROJECT\\...`); change the path constants at the top of each notebook before running. Raw data and model weights are **not** included in this repository.

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==================== تنظیمات — این دو خط رو حتماً چک/عوض کن ====================
METADATA_CSV = r"D:\NSCLC_PROJECT\dataset_metadata.csv"
RAW_DATA_ROOT = r"D:\NSCLC_PROJECT\raw-zip\NSCLC-Radiogenomics"   # <-- مسیر ریشه‌ی فایل‌های خام (DICOM/NIfTI)
OUTPUT_CSV = r"D:\NSCLC_PROJECT\spacing_metadata.csv"
# ===================================================================================


def find_patient_root(clean_id, root):
    """پوشه یا فایل مربوط به این بیمار رو زیر root پیدا می‌کنه."""
    root = Path(root)

    candidate_dir = root / clean_id
    if candidate_dir.is_dir():
        return candidate_dir, "dir"

    for ext in [".nii.gz", ".nii"]:
        candidate_file = root / f"{clean_id}{ext}"
        if candidate_file.exists():
            return candidate_file, "nifti_file"

    # جستجوی بازگشتی اگه ساختار پوشه‌ها تودرتوتره
    matches = list(root.rglob(f"*{clean_id}*"))
    dirs = [m for m in matches if m.is_dir()]
    files = [m for m in matches if m.suffix in (".nii", ".gz")]
    if dirs:
        return dirs[0], "dir"
    if files:
        return files[0], "nifti_file"

    return None, None


def find_ct_series_dir(patient_root):
    """
    زیر پوشه بیمار می‌گرده و پوشه‌ای که Modality=CT داره و بیشترین
    تعداد فایل .dcm رو داره برمی‌گردونه (سری اصلی CT، نه RTSTRUCT/PET/...).
    """
    import pydicom

    best_dir, best_count = None, 0
    for d in Path(patient_root).rglob("*"):
        if not d.is_dir():
            continue
        dcm_files = list(d.glob("*.dcm"))
        if len(dcm_files) < 10:
            continue
        try:
            ds = pydicom.dcmread(str(dcm_files[0]), stop_before_pixels=True)
            modality = getattr(ds, "Modality", "")
        except Exception:
            continue
        if modality == "CT" and len(dcm_files) > best_count:
            best_count, best_dir = len(dcm_files), d

    return best_dir


def get_dicom_spacing(dicom_dir):
    import pydicom

    dcm_files = sorted(Path(dicom_dir).glob("*.dcm"))
    slices = []
    for f in dcm_files:
        try:
            slices.append(pydicom.dcmread(str(f), stop_before_pixels=True))
        except Exception:
            continue
    if not slices:
        return None

    first = slices[0]
    row_spacing, col_spacing = map(float, first.PixelSpacing)  # (row=y, col=x)
    slice_thickness_tag = float(getattr(first, "SliceThickness", np.nan))

    z_positions = [float(ds.ImagePositionPatient[2]) for ds in slices
                   if hasattr(ds, "ImagePositionPatient")]
    if len(z_positions) >= 2:
        z_positions.sort()
        z_spacing = float(np.median(np.diff(z_positions)))
    else:
        z_spacing = slice_thickness_tag

    return {
        "spacing_x": col_spacing,
        "spacing_y": row_spacing,
        "spacing_z": z_spacing,
        "slice_thickness_tag": slice_thickness_tag,
        "n_slices_found": len(slices),
    }


def get_nifti_spacing(nifti_path):
    import nibabel as nib

    img = nib.load(str(nifti_path))
    zx, zy, zz = img.header.get_zooms()[:3]
    return {
        "spacing_x": float(zx),
        "spacing_y": float(zy),
        "spacing_z": float(zz),
        "slice_thickness_tag": float(zz),
        "n_slices_found": img.shape[2] if len(img.shape) > 2 else None,
    }


def main():
    df = pd.read_csv(METADATA_CSV)
    records = []

    for _, row in df.iterrows():
        clean_id = row["clean_id"]
        path, kind = find_patient_root(clean_id, RAW_DATA_ROOT)

        if path is None:
            print(f"⚠️  مسیر خام برای {clean_id} پیدا نشد")
            records.append({"clean_id": clean_id, "source_format": "NOT_FOUND"})
            continue

        try:
            if kind == "dir":
                ct_dir = find_ct_series_dir(path)
                if ct_dir is None:
                    print(f"⚠️  سری CT برای {clean_id} پیدا نشد (زیر {path})")
                    records.append({"clean_id": clean_id, "source_format": "NO_CT_SERIES"})
                    continue
                info = get_dicom_spacing(ct_dir)
                fmt = "dicom"
            else:
                info = get_nifti_spacing(path)
                fmt = "nifti"

            if info is None:
                print(f"⚠️  نتونستم spacing رو برای {clean_id} بخونم")
                continue

            info["clean_id"] = clean_id
            info["source_format"] = fmt
            records.append(info)
            print(f"✅ {clean_id}: spacing(x,y,z) = "
                  f"({info['spacing_x']:.3f}, {info['spacing_y']:.3f}, {info['spacing_z']:.3f}) mm  [{fmt}]")

        except Exception as e:
            print(f"❌ خطا برای {clean_id}: {e}")
            records.append({"clean_id": clean_id, "source_format": f"ERROR: {e}"})

    result = pd.DataFrame(records)
    result.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ ذخیره شد: {OUTPUT_CSV}  ({result['spacing_z'].notna().sum() if 'spacing_z' in result else 0} / {len(df)} بیمار موفق)")

    if "spacing_z" in result.columns:
        merged = df.merge(result, on="clean_id", how="left")
        print("\n--- مقایسه‌ی خیلی مهم: spacing_z به تفکیک لیبل (تست confound پروتکل) ---")
        print(merged.groupby("label")["spacing_z"].describe())
        print("\n--- spacing_z به تفکیک سایت (AMC vs R01) ---")
        merged["site"] = merged["clean_id"].str.split("-").str[0]
        print(merged.groupby("site")["spacing_z"].describe())


if __name__ == "__main__":
    main()


✅ AMC-001: spacing(x,y,z) = (0.977, 0.977, 3.270) mm  [dicom]
✅ AMC-002: spacing(x,y,z) = (0.549, 0.549, 1.000) mm  [dicom]
✅ AMC-003: spacing(x,y,z) = (0.533, 0.533, 1.000) mm  [dicom]
✅ AMC-004: spacing(x,y,z) = (0.713, 0.713, 1.000) mm  [dicom]
✅ AMC-005: spacing(x,y,z) = (0.648, 0.648, 1.000) mm  [dicom]
✅ AMC-006: spacing(x,y,z) = (0.691, 0.691, 1.000) mm  [dicom]
✅ AMC-007: spacing(x,y,z) = (0.650, 0.650, 1.000) mm  [dicom]
✅ AMC-009: spacing(x,y,z) = (0.787, 0.787, 1.000) mm  [dicom]
✅ AMC-010: spacing(x,y,z) = (0.652, 0.652, 1.000) mm  [dicom]
✅ AMC-011: spacing(x,y,z) = (0.842, 0.842, 1.000) mm  [dicom]
✅ AMC-012: spacing(x,y,z) = (0.607, 0.607, 0.800) mm  [dicom]
✅ AMC-013: spacing(x,y,z) = (0.703, 0.703, 1.250) mm  [dicom]
✅ AMC-014: spacing(x,y,z) = (0.746, 0.746, 1.000) mm  [dicom]
✅ AMC-015: spacing(x,y,z) = (0.701, 0.701, 0.700) mm  [dicom]
✅ AMC-016: spacing(x,y,z) = (0.803, 0.803, 0.700) mm  [dicom]
✅ AMC-017: spacing(x,y,z) = (0.814, 0.814, 1.000) mm  [dicom]
✅ AMC-01

In [19]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, balanced_accuracy_score

# ==================== تنظیمات نهایی ====================
METADATA_CSV = r"D:\NSCLC_PROJECT\dataset_metadata.csv"
EXCLUDE_IDS = ["R01-091", "R01-133"]          
N_SLICES_PER_PATIENT = 6                        
IMG_SIZE = 224                                  
BATCH_SIZE = 8
NUM_EPOCHS = 10                                 
LR = 5e-5                                       
MODEL_NAME = "swin_tiny_patch4_window7_224"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
# ===================================================

NUM_WORKERS = 2
if DEVICE.type == "cpu":
    import os
    torch.set_num_threads(os.cpu_count())
    NUM_WORKERS = 0

torch.manual_seed(SEED)
np.random.seed(SEED)

def select_informative_slices(volume, k):
    nonzero_per_slice = (volume > 0).sum(axis=(1, 2))
    k = min(k, len(nonzero_per_slice))
    top_k_idx = np.argsort(nonzero_per_slice)[-k:]
    return np.sort(top_k_idx)

def precompute_slice_selection(df, n_slices):
    selection = {}
    print("در حال انتخاب اسلایس‌های اطلاعاتی هر بیمار ...")
    for _, row in df.iterrows():
        volume = np.load(row["file_path"], mmap_mode="r")
        selection[row["clean_id"]] = select_informative_slices(volume, n_slices)
    print("انتخاب اسلایس‌ها به اتمام رسید.\n")
    return selection

class CTSliceDataset(Dataset):
    def __init__(self, df, slice_selection, img_size, augment=False):
        self.samples = [] 
        self.img_size = img_size
        self.augment = augment

        for _, row in df.iterrows():
            for idx in slice_selection[row["clean_id"]]:
                self.samples.append((row["file_path"], int(idx), int(row["label"]), row["clean_id"]))

        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        file_path, slice_idx, label, clean_id = self.samples[i]
        volume = np.load(file_path, mmap_mode="r")
        img = np.array(volume[slice_idx]).astype(np.float32)

        img_t = torch.from_numpy(img).unsqueeze(0)
        img_t = torch.nn.functional.interpolate(
            img_t.unsqueeze(0), size=(self.img_size, self.img_size),
            mode="bilinear", align_corners=False
        ).squeeze(0)
        img_t = img_t.repeat(3, 1, 1)

        if self.augment:
            if torch.rand(1).item() < 0.5:
                img_t = torch.flip(img_t, dims=[2])
            angle = float(torch.empty(1).uniform_(-15, 15))
            img_t = TF.rotate(img_t, angle)

        img_t = (img_t - self.mean) / self.std
        return img_t, label, clean_id

def build_model():
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=2, drop_rate=0.3, drop_path_rate=0.2)
    for name, param in model.named_parameters():
        if "layers.2" in name or "layers.3" in name or "head" in name or "norm" in name:
            param.requires_grad = True
        else:
            param.requires_grad = False
    return model.to(DEVICE)

def evaluate_patient_level(model, loader):
    model.eval()
    all_probs, all_labels, all_ids = [], [], []
    with torch.no_grad():
        for imgs, labels, ids in loader:
            imgs = imgs.to(DEVICE)
            probs = torch.softmax(model(imgs), dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())
            all_ids.extend(ids)

    df = pd.DataFrame({"clean_id": all_ids, "prob": all_probs, "label": all_labels})
    patient_df = df.groupby("clean_id").agg(prob=("prob", "mean"), label=("label", "first")).reset_index()
    return patient_df

def main():
    df = pd.read_csv(METADATA_CSV)
    df = df[~df["clean_id"].isin(EXCLUDE_IDS)].reset_index(drop=True)
    print(f"تعداد کل بیمار پس از حذف نمونه‌های خراب: {len(df)}")

    slice_selection = precompute_slice_selection(df, N_SLICES_PER_PATIENT)
    fold_list = sorted(df["fold"].unique())
    
    all_fold_metrics = []
    oof_predictions = [] # برای نگهداری پیش‌بینی‌های Out-Of-Fold جهت آنسمبل نهایی

    for fold_num in fold_list:
        train_df = df[df["fold"] != fold_num]
        val_df = df[df["fold"] == fold_num]
        print(f"\n--- آموزش و ارزیابی Fold {fold_num} (Train: {len(train_df)}, Val: {len(val_df)}) ---")

        train_ds = CTSliceDataset(train_df, slice_selection, IMG_SIZE, augment=True)
        val_ds = CTSliceDataset(val_df, slice_selection, IMG_SIZE, augment=False)

        labels_arr = np.array([s[2] for s in train_ds.samples])
        class_counts = np.bincount(labels_arr)
        class_weights = (1.0 / class_counts)
        sample_weights = class_weights[labels_arr]
        sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

        model = build_model()
        class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=class_weights_t)
        optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-3)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

        best_val_score, best_state = -1, None

        for epoch in range(NUM_EPOCHS):
            model.train()
            total_loss = 0.0
            for imgs, labels, _ in train_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(model(imgs), labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * imgs.size(0)
            scheduler.step()

            val_patient_df = evaluate_patient_level(model, val_loader)
            auc = roc_auc_score(val_patient_df["label"], val_patient_df["prob"]) if val_patient_df["label"].nunique() > 1 else 0.5
            preds = (val_patient_df["prob"] >= 0.5).astype(int)
            bal_acc = balanced_accuracy_score(val_patient_df["label"], preds)
            score = bal_acc + auc

            if score > best_val_score:
                best_val_score = score
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if best_state is not None:
            model.load_state_dict(best_state)
            
        final_val_df = evaluate_patient_level(model, val_loader)
        final_val_df["fold"] = fold_num
        oof_predictions.append(final_val_df)

        preds = (final_val_df["prob"] >= 0.5).astype(int)
        acc = accuracy_score(final_val_df["label"], preds)
        bal_acc = balanced_accuracy_score(final_val_df["label"], preds)
        auc = roc_auc_score(final_val_df["label"], final_val_df["prob"])
        f1 = f1_score(final_val_df["label"], preds, zero_division=0)

        all_fold_metrics.append({"fold": fold_num, "accuracy": acc, "balanced_accuracy": bal_acc, "auc": auc, "f1": f1})
        print(f"نتیجه نهایی Fold {fold_num} -> Acc: {acc:.3f} | Balanced Acc: {bal_acc:.3f} | AUC: {auc:.3f} | F1: {f1:.3f}")

    results_df = pd.DataFrame(all_fold_metrics)
    results_df.to_csv("swin_cv_results_final.csv", index=False)
    
    # ارزیابی تجمیعی کل (Out-Of-Fold Ensemble Evaluation)
    oof_df = pd.concat(oof_predictions).reset_index(drop=True)
    oof_preds = (oof_df["prob"] >= 0.5).astype(int)
    
    overall_acc = accuracy_score(oof_df["label"], oof_preds)
    overall_bal_acc = balanced_accuracy_score(oof_df["label"], oof_preds)
    overall_auc = roc_auc_score(oof_df["label"], oof_df["prob"])
    overall_f1 = f1_score(oof_df["label"], oof_preds, zero_division=0)

    print("\n========== خلاصه نهایی Cross-Validation (میانگین فولدها) ==========")
    for col in ["accuracy", "balanced_accuracy", "auc", "f1"]:
        print(f"{col}: {results_df[col].mean():.3f} ± {results_df[col].std():.3f}")

    print("\n========== عملکرد کل سیستم (OOF Ensemble Overall) ==========")
    print(f"Overall Accuracy         : {overall_acc:.3f}")
    print(f"Overall Balanced Accuracy: {overall_bal_acc:.3f}")
    print(f"Overall AUC              : {overall_auc:.3f}")
    print(f"Overall F1-Score         : {overall_f1:.3f}")

if __name__ == "__main__":
    main()

تعداد کل بیمار پس از حذف نمونه‌های خراب: 170
در حال انتخاب اسلایس‌های اطلاعاتی هر بیمار ...
انتخاب اسلایس‌ها به اتمام رسید.


--- آموزش و ارزیابی Fold 0 (Train: 135, Val: 35) ---
نتیجه نهایی Fold 0 -> Acc: 0.629 | Balanced Acc: 0.605 | AUC: 0.509 | F1: 0.435

--- آموزش و ارزیابی Fold 1 (Train: 135, Val: 35) ---
نتیجه نهایی Fold 1 -> Acc: 0.714 | Balanced Acc: 0.626 | AUC: 0.620 | F1: 0.444

--- آموزش و ارزیابی Fold 2 (Train: 136, Val: 34) ---
نتیجه نهایی Fold 2 -> Acc: 0.706 | Balanced Acc: 0.635 | AUC: 0.577 | F1: 0.444

--- آموزش و ارزیابی Fold 3 (Train: 137, Val: 33) ---
نتیجه نهایی Fold 3 -> Acc: 0.758 | Balanced Acc: 0.670 | AUC: 0.785 | F1: 0.500

--- آموزش و ارزیابی Fold 4 (Train: 137, Val: 33) ---
نتیجه نهایی Fold 4 -> Acc: 0.788 | Balanced Acc: 0.750 | AUC: 0.708 | F1: 0.632
